In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# Install required packages
!pip install -q sentence-transformers faiss-cpu

# Prevent tokenizer warnings
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("✅ All packages installed")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's de

In [3]:
import json
import pandas as pd
import os

# Path to your MedQA dataset
base_path = "/kaggle/input/medqamini-project"

# Load US test questions
def load_medqa_questions(split='test'):
    questions = []
    file_path = f"{base_path}/data_clean/questions/US/{split}.jsonl"
    
    with open(file_path, 'r') as f:
        for line in f:
            questions.append(json.loads(line))
    
    return questions

# Load all splits
train_questions = load_medqa_questions('train')
dev_questions = load_medqa_questions('dev')
test_questions = load_medqa_questions('test')

print(f"✅ Loaded MedQA Questions:")
print(f"  Train: {len(train_questions)}")
print(f"  Dev: {len(dev_questions)}")
print(f"  Test: {len(test_questions)}")

# Show sample question
print("\n📋 Sample Question:")
sample = test_questions[0]
print(f"Question: {sample['question']}")
print(f"Options: {sample['options']}")
print(f"Answer: {sample['answer']}")


✅ Loaded MedQA Questions:
  Train: 10178
  Dev: 1272
  Test: 1273

📋 Sample Question:
Question: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Options: {'A': 'Disclose the error to the patient but leave it out of the operative report', 'B': 'Disclose the error to the patient and put it in the operative report', 'C': 'Tell the attending that he cannot fail to disclose this mistake', 'D': 'Report the physician to the ethics committee', 

In [4]:
import glob

# Path to textbooks
textbook_path = f"{base_path}/data_clean/textbooks/en"

# Get all textbook files
textbook_files = glob.glob(f"{textbook_path}/*.txt")

print(f"✅ Found {len(textbook_files)} textbook files")

# Load textbook content
textbooks = []
for file_path in textbook_files[:10]:  # Start with first 10 for testing
    filename = os.path.basename(file_path)
    
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    
    textbooks.append({
        'doc_id': filename.replace('.txt', ''),
        'title': filename,
        'text': text,
        'length': len(text)
    })

textbooks_df = pd.DataFrame(textbooks)

print(f"\n📊 Textbook Statistics:")
print(f"  Total files loaded: {len(textbooks_df)}")
print(f"  Average length: {textbooks_df['length'].mean():.0f} chars")
print(f"  Total chars: {textbooks_df['length'].sum():,}")

# Show sample
print(f"\n📖 Sample from '{textbooks_df.iloc[0]['title']}':")
print(textbooks_df.iloc[0]['text'][:500])


✅ Found 18 textbook files

📊 Textbook Statistics:
  Total files loaded: 10
  Average length: 5362342 chars
  Total chars: 53,623,417

📖 Sample from 'First_Aid_Step1.txt':
“Biochemistry is the study of carbon compounds that crawl.” “We think we have found the basic mechanism by which life comes from life.” —Francis H. C. Crick “The biochemistry and biophysics are the notes required for life; they conspire, collectively, to generate the real unit of life, the organism.”

This high-yield material includes molecular biology, genetics, cell biology, and principles of metabolism (especially vitamins, cofactors, minerals, and single-enzyme-deficiency diseases). When stu


In [5]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks by words"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.strip()) > 100:  # Only keep substantial chunks
            chunks.append(chunk)
    
    return chunks

# Chunk all textbooks
all_chunks = []
for idx, row in textbooks_df.iterrows():
    chunks = chunk_text(row['text'], chunk_size=500, overlap=50)
    
    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            'doc_id': row['doc_id'],
            'title': row['title'],
            'chunk_id': chunk_id,
            'chunk': chunk,
            'chunk_length': len(chunk)
        })

chunks_df = pd.DataFrame(all_chunks)

print(f"✅ Created {len(chunks_df)} chunks from {len(textbooks_df)} textbooks")
print(f"📊 Average chunk length: {chunks_df['chunk_length'].mean():.0f} characters")
print(f"📊 Chunks per textbook: {len(chunks_df)/len(textbooks_df):.1f}")

# Save
chunks_df.to_csv('textbook_chunks.csv', index=False)
print("\n✅ Saved to 'textbook_chunks.csv'")

# Show sample chunk
print(f"\n📄 Sample Chunk:")
print(f"Source: {chunks_df.iloc[0]['title']}")
print(f"Text: {chunks_df.iloc[0]['chunk'][:300]}...")


✅ Created 17457 chunks from 10 textbooks
📊 Average chunk length: 3403 characters
📊 Chunks per textbook: 1745.7

✅ Saved to 'textbook_chunks.csv'

📄 Sample Chunk:
Source: First_Aid_Step1.txt
Text: “Biochemistry is the study of carbon compounds that crawl.” “We think we have found the basic mechanism by which life comes from life.” —Francis H. C. Crick “The biochemistry and biophysics are the notes required for life; they conspire, collectively, to generate the real unit of life, the organism....


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# Load embedding model
print("🔄 Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded: all-MiniLM-L6-v2")

# Generate embeddings
print(f"\n🔄 Generating embeddings for {len(chunks_df)} chunks...")
start_time = time.time()

chunk_texts = chunks_df['chunk'].tolist()
embeddings = model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

elapsed = time.time() - start_time

print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.2f}s")
print(f"📊 Embedding shape: {embeddings.shape}")
print(f"⚡ Speed: {len(embeddings)/elapsed:.1f} embeddings/sec")

# Save
np.save('textbook_embeddings.npy', embeddings)
print("✅ Saved to 'textbook_embeddings.npy'")


2026-01-02 06:57:40.528073: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767337060.698873      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767337060.744519      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: all-MiniLM-L6-v2

🔄 Generating embeddings for 17457 chunks...


Batches:   0%|          | 0/546 [00:00<?, ?it/s]


✅ Generated 17457 embeddings in 71.08s
📊 Embedding shape: (17457, 384)
⚡ Speed: 245.6 embeddings/sec
✅ Saved to 'textbook_embeddings.npy'


In [7]:
import faiss

# Load embeddings
embeddings = np.load('textbook_embeddings.npy')

print("🔄 Building FAISS index...")
dimension = embeddings.shape[1]

# Create index (cosine similarity via inner product after normalization)
index = faiss.IndexFlatIP(dimension)

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"✅ FAISS index created: {index.ntotal} vectors, {dimension}D")

# Save index
faiss.write_index(index, 'textbook_faiss.index')
print("✅ Saved to 'textbook_faiss.index'")


🔄 Building FAISS index...
✅ FAISS index created: 17457 vectors, 384D
✅ Saved to 'textbook_faiss.index'


In [8]:
# Load components
index = faiss.read_index('textbook_faiss.index')
chunks_df = pd.read_csv('textbook_chunks.csv')

print("🧪 TESTING RETRIEVAL")
print("="*80)

# Test queries
test_queries = [
    "What are the symptoms of lung cancer?",
    "How is diabetes treated?",
    "What causes heart disease?"
]

for query in test_queries:
    print(f"\n❓ Query: {query}")
    
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Search
    distances, indices = index.search(query_embedding, 3)
    
    print("📊 Top 3 Results:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        chunk_row = chunks_df.iloc[idx]
        print(f"\n  {i+1}. Similarity: {dist:.3f}")
        print(f"     Source: {chunk_row['title']}")
        print(f"     Text: {chunk_row['chunk'][:150]}...")
    
    print("─"*80)

print("\n✅ BASELINE RETRIEVAL WORKING!")


🧪 TESTING RETRIEVAL

❓ Query: What are the symptoms of lung cancer?
📊 Top 3 Results:

  1. Similarity: 0.602
     Source: InternalMed_Harrison.txt
     Text: obstructive pulmonary disease (COPD) age 40 years or older should prompt a thorough investigation for lung cancer even in the face of a normal CXR. A ...

  2. Similarity: 0.564
     Source: InternalMed_Harrison.txt
     Text: Possible inhalational exposures should be explored, including those at the work place (e.g., asbestos, wood smoke) and those associated with leisure (...

  3. Similarity: 0.511
     Source: InternalMed_Harrison.txt
     Text: smoke. The risks of cancer increase with the increasing number of cigarettes smoked per day and with increasing duration of smoking. Additionally, the...
────────────────────────────────────────────────────────────────────────────────

❓ Query: How is diabetes treated?
📊 Top 3 Results:

  1. Similarity: 0.573
     Source: Gynecology_Novak.txt
     Text: sulfonylureas) or other Table 9.

In [9]:
def extractive_rag(query, top_k=3):
    """Simple RAG without external LLM - just returns retrieved chunks"""
    
    # Encode query
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    # Retrieve
    distances, indices = index.search(query_embedding, top_k)
    
    # Collect results
    retrieved = []
    for idx, dist in zip(indices[0], distances[0]):
        chunk_data = chunks_df.iloc[idx]
        retrieved.append({
            'chunk': chunk_data['chunk'],
            'similarity': float(dist),
            'source': chunk_data['title'],
            'doc_id': chunk_data['doc_id']
        })
    
    # Create answer from retrieved evidence
    answer = "Based on retrieved medical textbook evidence:\n\n"
    for i, item in enumerate(retrieved):
        answer += f"[{i+1}] Source: {item['source']}\n"
        answer += f"    {item['chunk'][:400]}...\n\n"
    
    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': retrieved,
        'top_similarity': retrieved[0]['similarity'] if retrieved else 0
    }

print("✅ Extractive RAG ready (no external LLM needed)")


✅ Extractive RAG ready (no external LLM needed)


In [10]:
# Test the extractive RAG
test_queries = [
    "What are the symptoms of lung cancer?",
    "How is diabetes treated?",
    "What causes heart disease?"
]

print("\n" + "="*80)
print("🧪 TESTING EXTRACTIVE RAG")
print("="*80)

for query in test_queries:
    print(f"\n{'─'*80}")
    print(f"❓ Query: {query}")
    print("─"*40)
    
    result = extractive_rag(query, top_k=3)
    
    print(f"\n💬 Answer:\n{result['answer']}")
    print(f"📊 Top similarity score: {result['top_similarity']:.3f}")
    print("─"*80)

print("\n✅ EXTRACTIVE RAG SYSTEM WORKING!")



🧪 TESTING EXTRACTIVE RAG

────────────────────────────────────────────────────────────────────────────────
❓ Query: What are the symptoms of lung cancer?
────────────────────────────────────────

💬 Answer:
Based on retrieved medical textbook evidence:

[1] Source: InternalMed_Harrison.txt
    obstructive pulmonary disease (COPD) age 40 years or older should prompt a thorough investigation for lung cancer even in the face of a normal CXR. A persistent pneumonia without constitutional symptoms and unresponsive to repeated courses of antibiotics also should prompt an evaluation for the underlying cause. Lung cancer arising in a lifetime never smoker is more common in women and East Asians...

[2] Source: InternalMed_Harrison.txt
    Possible inhalational exposures should be explored, including those at the work place (e.g., asbestos, wood smoke) and those associated with leisure (e.g., excrement from pet birds) (Chap. 311). Travel predisposes to certain infections of the respiratory trac

In [11]:
import re

def classify_query_complexity(question):
    """
    Classify query into SIMPLE/MEDIUM/COMPLEX
    Based on: length, keywords, structure
    """
    q_lower = question.lower()
    words = question.split()
    word_count = len(words)
    
    # Complex indicators
    complex_keywords = [
        'compare', 'difference between', 'distinguish',
        'explain how', 'explain why', 'mechanism',
        'pathophysiology', 'differential diagnosis',
        'contraindication', 'workup', 'step-by-step'
    ]
    
    # Simple indicators
    simple_patterns = [
        question.strip().startswith('What is'),
        question.strip().startswith('What are'),
        question.strip().startswith('Define'),
        word_count <= 8
    ]
    
    # Check for complex patterns
    has_complex = any(keyword in q_lower for keyword in complex_keywords)
    has_multiple_clauses = q_lower.count(' and ') >= 2 or q_lower.count(',') >= 2
    
    # Classification logic
    if has_complex or word_count > 20 or has_multiple_clauses:
        return 'COMPLEX'
    elif any(simple_patterns):
        return 'SIMPLE'
    else:
        return 'MEDIUM'

# Test the router
test_queries = [
    "What is diabetes?",
    "What are the symptoms of lung cancer?",
    "How is hypertension treated?",
    "Compare the pathophysiology of Type 1 and Type 2 diabetes",
    "Explain the mechanism of action of ACE inhibitors and their contraindications",
    "What is the differential diagnosis for chest pain?"
]

print("🔄 Testing Query Complexity Classifier")
print("="*80)

for query in test_queries:
    complexity = classify_query_complexity(query)
    print(f"\n{complexity:8} | {query}")

print("\n✅ Query router ready")


🔄 Testing Query Complexity Classifier

SIMPLE   | What is diabetes?

SIMPLE   | What are the symptoms of lung cancer?

SIMPLE   | How is hypertension treated?

COMPLEX  | Compare the pathophysiology of Type 1 and Type 2 diabetes

COMPLEX  | Explain the mechanism of action of ACE inhibitors and their contraindications

COMPLEX  | What is the differential diagnosis for chest pain?

✅ Query router ready


In [12]:
def detect_knowledge_gap(query, top_k=5, threshold=0.45):
    """
    Detect if KB has sufficient knowledge to answer query
    Returns: (has_knowledge: bool, confidence: float, explanation: str)
    """
    
    # Encode and retrieve
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)
    
    # Retrieval signals
    top1_score = float(distances[0][0]) if len(distances[0]) > 0 else 0.0
    mean_score = float(np.mean(distances[0])) if len(distances[0]) > 0 else 0.0
    score_gap = float(distances[0][0] - distances[0][1]) if len(distances[0]) > 1 else 0.0
    
    # Decision logic
    has_knowledge = top1_score >= threshold
    
    # Explanation
    if not has_knowledge:
        if top1_score < 0.3:
            explanation = f"Very low similarity (top1={top1_score:.3f}). Query likely outside knowledge base scope."
        elif top1_score < threshold:
            explanation = f"Below confidence threshold (top1={top1_score:.3f} < {threshold}). Insufficient evidence."
        else:
            explanation = f"Borderline confidence (top1={top1_score:.3f})"
    else:
        explanation = f"Sufficient evidence found (top1={top1_score:.3f}, mean={mean_score:.3f})"
    
    return {
        'has_knowledge': has_knowledge,
        'top1_similarity': top1_score,
        'mean_similarity': mean_score,
        'score_gap': score_gap,
        'explanation': explanation
    }

print("✅ Knowledge gap detector ready")


✅ Knowledge gap detector ready


In [13]:
def adaptive_rag(query):
    """
    Complete Adaptive RAG with:
    1. Query complexity routing
    2. Knowledge gap detection
    3. Adaptive retrieval
    4. Safe answer generation
    """
    
    # Step 1: Classify complexity
    complexity = classify_query_complexity(query)
    
    # Step 2: Adaptive retrieval parameters
    if complexity == 'SIMPLE':
        top_k = 2
        threshold = 0.40
    elif complexity == 'MEDIUM':
        top_k = 3
        threshold = 0.45
    else:  # COMPLEX
        top_k = 5
        threshold = 0.50
    
    # Step 3: Knowledge gap detection
    gap_result = detect_knowledge_gap(query, top_k=top_k, threshold=threshold)
    
    # Step 4: Decide whether to answer
    if not gap_result['has_knowledge']:
        return {
            'query': query,
            'complexity': complexity,
            'gap_detected': True,
            'gap_info': gap_result,
            'answer': "❌ KNOWLEDGE GAP DETECTED\n\n" +
                     f"{gap_result['explanation']}\n\n" +
                     "This question cannot be answered reliably with the current knowledge base. " +
                     "Please consult a qualified medical professional or refer to additional sources.",
            'retrieved_chunks': [],
            'top_similarity': gap_result['top1_similarity']
        }
    
    # Step 5: Retrieve evidence
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)
    
    retrieved = []
    for idx, dist in zip(indices[0], distances[0]):
        chunk_data = chunks_df.iloc[idx]
        retrieved.append({
            'chunk': chunk_data['chunk'],
            'similarity': float(dist),
            'source': chunk_data['title'],
            'doc_id': chunk_data['doc_id']
        })
    
    # Step 6: Generate answer from evidence
    answer = f"✅ ANSWER (Confidence: {gap_result['top1_similarity']:.2f})\n\n"
    answer += "Based on retrieved medical textbook evidence:\n\n"
    
    for i, item in enumerate(retrieved):
        answer += f"[{i+1}] Source: {item['source']} (similarity: {item['similarity']:.3f})\n"
        answer += f"    {item['chunk'][:350]}...\n\n"
    
    return {
        'query': query,
        'complexity': complexity,
        'gap_detected': False,
        'gap_info': gap_result,
        'answer': answer,
        'retrieved_chunks': retrieved,
        'top_similarity': gap_result['top1_similarity'],
        'num_chunks_used': top_k
    }

print("✅ Adaptive RAG pipeline ready!")


✅ Adaptive RAG pipeline ready!


In [14]:
print("\n" + "="*80)
print("🧪 TESTING ADAPTIVE RAG - IN-DOMAIN MEDICAL QUESTIONS")
print("="*80)

in_domain_queries = [
    "What is diabetes?",
    "What are the symptoms of lung cancer?",
    "How is hypertension treated?",
    "Compare Type 1 and Type 2 diabetes"
]

for query in in_domain_queries:
    print(f"\n{'─'*80}")
    print(f"❓ Query: {query}")
    print("─"*40)
    
    result = adaptive_rag(query)
    
    print(f"🏷️  Complexity: {result['complexity']}")
    print(f"📊 Top Similarity: {result['top_similarity']:.3f}")
    print(f"🚦 Gap Detected: {result['gap_detected']}")
    print(f"📚 Chunks Used: {result['num_chunks_used']}")
    
    print(f"\n💬 Answer Preview:")
    print(result['answer'][:600])
    if len(result['answer']) > 600:
        print("...")
    
    print("─"*80)

print("\n✅ IN-DOMAIN TESTING COMPLETE!")



🧪 TESTING ADAPTIVE RAG - IN-DOMAIN MEDICAL QUESTIONS

────────────────────────────────────────────────────────────────────────────────
❓ Query: What is diabetes?
────────────────────────────────────────
🏷️  Complexity: SIMPLE
📊 Top Similarity: 0.602
🚦 Gap Detected: False
📚 Chunks Used: 2

💬 Answer Preview:
✅ ANSWER (Confidence: 0.60)

Based on retrieved medical textbook evidence:

[1] Source: Biochemistry_Lippincott.txt (similarity: 0.602)
    molecules. There is currently no preventative treatment for T1D. The risk for T2D can be significantly decreased by a combined regimen of medical nutrition therapy, weight loss, exercise, and aggressive control of hypertension and dyslipidemias. For example, Figure 25.13 shows the incidence of disease in normal and overweight individuals with varyi...

[2] Source: Gynecology_Novak.txt (similarity: 0.542)
    a result of gluconeogenesis and ketogenesis in t
...
────────────────────────────────────────────────────────────────────────────────

────

In [15]:
print("\n" + "="*80)
print("🧪 TESTING KNOWLEDGE GAP DETECTION - OUT-OF-DOMAIN QUESTIONS")
print("="*80)

out_of_domain_queries = [
    "What is the weather forecast for tomorrow?",
    "Who won the 2024 presidential election?",
    "How do I fix my laptop screen?",
    "What are the best restaurants in New York?",
    "Explain quantum computing",
    "What is the treatment for Martian flu?"  # Fictional disease
]

gap_results = []

for query in out_of_domain_queries:
    print(f"\n{'─'*80}")
    print(f"❓ Query: {query}")
    print("─"*40)
    
    result = adaptive_rag(query)
    
    print(f"🏷️  Complexity: {result['complexity']}")
    print(f"📊 Top Similarity: {result['top_similarity']:.3f}")
    print(f"🚦 Gap Detected: {result['gap_detected']}")
    
    if result['gap_detected']:
        print(f"✅ CORRECTLY REFUSED TO ANSWER")
        print(f"📝 Reason: {result['gap_info']['explanation']}")
    else:
        print(f"⚠️  WARNING: Attempted to answer (may be hallucination)")
    
    gap_results.append({
        'query': query,
        'gap_detected': result['gap_detected'],
        'top_similarity': result['top_similarity']
    })
    
    print("─"*80)

# Summary
gap_df = pd.DataFrame(gap_results)
detection_rate = gap_df['gap_detected'].sum() / len(gap_df) * 100

print(f"\n📊 KNOWLEDGE GAP DETECTION SUMMARY:")
print(f"  Out-of-domain queries: {len(gap_df)}")
print(f"  Gaps detected: {gap_df['gap_detected'].sum()}")
print(f"  Detection rate: {detection_rate:.1f}%")

print("\n✅ KNOWLEDGE GAP DETECTION WORKING!")



🧪 TESTING KNOWLEDGE GAP DETECTION - OUT-OF-DOMAIN QUESTIONS

────────────────────────────────────────────────────────────────────────────────
❓ Query: What is the weather forecast for tomorrow?
────────────────────────────────────────
🏷️  Complexity: SIMPLE
📊 Top Similarity: 0.418
🚦 Gap Detected: False
⚠️  WARNING: Attempted to answer (may be hallucination)
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
❓ Query: Who won the 2024 presidential election?
────────────────────────────────────────
🏷️  Complexity: SIMPLE
📊 Top Similarity: 0.219
🚦 Gap Detected: True
✅ CORRECTLY REFUSED TO ANSWER
📝 Reason: Very low similarity (top1=0.219). Query likely outside knowledge base scope.
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
❓ Query: How do I fix my laptop screen

In [17]:
# Load MedQA test questions for proper evaluation
test_questions = load_medqa_questions('test')

# Sample 50 questions for quick evaluation
eval_sample = test_questions[:50]

eval_results = []

print("🔄 Evaluating on MedQA test questions...")
print("="*80)

for i, qa in enumerate(eval_sample[:10]):  # Show first 10
    query = qa['question']
    
    result = adaptive_rag(query)
    
    eval_results.append({
        'question': query,
        'complexity': result['complexity'],
        'gap_detected': result['gap_detected'],
        'top_similarity': result['top_similarity'],
        'num_chunks': result.get('num_chunks_used', 0)  # Fixed: use .get() with default
    })
    
    if i < 3:  # Show first 3 examples
        print(f"\n{i+1}. {query[:80]}...")
        print(f"   Complexity: {result['complexity']} | Gap: {result['gap_detected']} | Sim: {result['top_similarity']:.3f}")

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv('evaluation_results.csv', index=False)

print(f"\n✅ Evaluated {len(eval_results)} questions")
print(f"📊 Complexity distribution:")
print(eval_df['complexity'].value_counts())
print(f"\n📊 Average similarity by complexity:")
print(eval_df.groupby('complexity')['top_similarity'].mean())

print("\n✅ Saved to 'evaluation_results.csv'")


🔄 Evaluating on MedQA test questions...

1. A junior orthopaedic surgery resident is completing a carpal tunnel repair with ...
   Complexity: COMPLEX | Gap: True | Sim: 0.421

2. A 67-year-old man with transitional cell carcinoma of the bladder comes to the p...
   Complexity: COMPLEX | Gap: False | Sim: 0.519

3. Two weeks after undergoing an emergency cardiac catherization with stenting for ...
   Complexity: COMPLEX | Gap: False | Sim: 0.568

✅ Evaluated 10 questions
📊 Complexity distribution:
complexity
COMPLEX    10
Name: count, dtype: int64

📊 Average similarity by complexity:
complexity
COMPLEX    0.546804
Name: top_similarity, dtype: float64

✅ Saved to 'evaluation_results.csv'


In [18]:
import matplotlib.pyplot as plt

# Load evaluation results
eval_df = pd.read_csv('evaluation_results.csv')

print("="*80)
print("📊 FINAL PROJECT SUMMARY")
print("="*80)

# 1. Overall statistics
print(f"\n📈 SYSTEM PERFORMANCE:")
print(f"  Total queries evaluated: {len(eval_df)}")
print(f"  Queries answered: {(~eval_df['gap_detected']).sum()}")
print(f"  Gaps detected: {eval_df['gap_detected'].sum()}")
print(f"  Gap detection rate: {eval_df['gap_detected'].sum()/len(eval_df)*100:.1f}%")

# 2. Complexity distribution
print(f"\n🏷️  QUERY COMPLEXITY DISTRIBUTION:")
complexity_dist = eval_df['complexity'].value_counts()
for complexity, count in complexity_dist.items():
    print(f"  {complexity:8}: {count:3} ({count/len(eval_df)*100:.1f}%)")

# 3. Similarity scores by complexity
print(f"\n📊 RETRIEVAL SIMILARITY BY COMPLEXITY:")
sim_by_complexity = eval_df.groupby('complexity')['top_similarity'].agg(['mean', 'min', 'max'])
print(sim_by_complexity)

# 4. Chunks used by complexity (excluding gaps)
answered_df = eval_df[~eval_df['gap_detected']]
if len(answered_df) > 0:
    print(f"\n📚 CHUNKS USED (when answering):")
    chunks_by_complexity = answered_df.groupby('complexity')['num_chunks'].mean()
    print(chunks_by_complexity)

print("\n" + "="*80)
print("✅ PROJECT IMPLEMENTATION COMPLETE!")
print("="*80)


📊 FINAL PROJECT SUMMARY

📈 SYSTEM PERFORMANCE:
  Total queries evaluated: 10
  Queries answered: 8
  Gaps detected: 2
  Gap detection rate: 20.0%

🏷️  QUERY COMPLEXITY DISTRIBUTION:
  COMPLEX :  10 (100.0%)

📊 RETRIEVAL SIMILARITY BY COMPLEXITY:
                mean       min       max
complexity                              
COMPLEX     0.546804  0.421056  0.615786

📚 CHUNKS USED (when answering):
complexity
COMPLEX    5.0
Name: num_chunks, dtype: float64

✅ PROJECT IMPLEMENTATION COMPLETE!


In [19]:
print("\n" + "="*80)
print("📋 DEMO EXAMPLES FOR PRESENTATION")
print("="*80)

# Demo 1: Simple query - should answer
print("\n" + "─"*80)
print("DEMO 1: SIMPLE QUERY (Definition)")
print("─"*80)
demo1 = adaptive_rag("What is hypertension?")
print(f"Query: {demo1['query']}")
print(f"Complexity: {demo1['complexity']}")
print(f"Gap Detected: {demo1['gap_detected']}")
print(f"Top Similarity: {demo1['top_similarity']:.3f}")
print(f"\nAnswer Preview:\n{demo1['answer'][:500]}...")

# Demo 2: Complex query - should answer with more context
print("\n" + "─"*80)
print("DEMO 2: COMPLEX QUERY (Comparison)")
print("─"*80)
demo2 = adaptive_rag("Compare the pathophysiology of Type 1 and Type 2 diabetes")
print(f"Query: {demo2['query']}")
print(f"Complexity: {demo2['complexity']}")
print(f"Gap Detected: {demo2['gap_detected']}")
print(f"Top Similarity: {demo2['top_similarity']:.3f}")
print(f"Chunks Used: {demo2.get('num_chunks_used', 0)}")

# Demo 3: Out-of-domain - should detect gap
print("\n" + "─"*80)
print("DEMO 3: OUT-OF-DOMAIN QUERY (Gap Detection)")
print("─"*80)
demo3 = adaptive_rag("How do I fix my computer's blue screen error?")
print(f"Query: {demo3['query']}")
print(f"Complexity: {demo3['complexity']}")
print(f"Gap Detected: {demo3['gap_detected']} ✅")
print(f"Top Similarity: {demo3['top_similarity']:.3f}")
print(f"\nSystem Response:\n{demo3['answer']}")

# Demo 4: Medical but rare/not in KB - should detect gap
print("\n" + "─"*80)
print("DEMO 4: RARE MEDICAL QUERY (Gap Detection)")
print("─"*80)
demo4 = adaptive_rag("What is the treatment for Ondine's curse syndrome?")
print(f"Query: {demo4['query']}")
print(f"Complexity: {demo4['complexity']}")
print(f"Gap Detected: {demo4['gap_detected']}")
print(f"Top Similarity: {demo4['top_similarity']:.3f}")

print("\n✅ Copy these examples to your PPT!")



📋 DEMO EXAMPLES FOR PRESENTATION

────────────────────────────────────────────────────────────────────────────────
DEMO 1: SIMPLE QUERY (Definition)
────────────────────────────────────────────────────────────────────────────────
Query: What is hypertension?
Complexity: SIMPLE
Gap Detected: False
Top Similarity: 0.592

Answer Preview:
✅ ANSWER (Confidence: 0.59)

Based on retrieved medical textbook evidence:

[1] Source: InternalMed_Harrison.txt (similarity: 0.592)
    with permission by Dr. Andrew C. Eisenhauer.) CHAPTER 297e Atlas of Percutaneous Revascularization Hypertensive vascular Disease Theodore A. Kotchen Hypertension is one of the leading causes of the global burden of disease. Approximately 7.6 million deaths (13–15% of the total) and 92 million disability-adjusted life years worldwide were attributab...

[2] Sour...

────────────────────────────────────────────────────────────────────────────────
DEMO 2: COMPLEX QUERY (Comparison)
─────────────────────────────────────────

In [21]:
# Save all important results
import json

# System configuration
config = {
    'embedding_model': 'all-MiniLM-L6-v2',
    'vector_store': 'FAISS (IndexFlatIP)',
    'num_textbooks': len(textbooks_df),
    'num_chunks': len(chunks_df),
    'embedding_dim': 384,
    'complexity_thresholds': {
        'SIMPLE': {'top_k': 2, 'threshold': 0.40},
        'MEDIUM': {'top_k': 3, 'threshold': 0.45},
        'COMPLEX': {'top_k': 5, 'threshold': 0.50}
    }
}

with open('system_config.json', 'w') as f:
    json.dump(config, f, indent=2)  # Fixed: dump (not dumps)

print("✅ Saved system configuration to 'system_config.json'")

# Save chunks info
chunks_info = {
    'total_chunks': len(chunks_df),
    'avg_chunk_length': int(chunks_df['chunk_length'].mean()),
    'textbooks_used': textbooks_df['title'].tolist()
}

with open('kb_info.json', 'w') as f:
    json.dump(chunks_info, f, indent=2)  # Fixed: dump (not dumps)

print("✅ Saved KB info to 'kb_info.json'")

# Summary for report
print(f"\n📄 SUMMARY FOR YOUR REPORT:")
print(f"="*80)
print(f"Knowledge Base:")
print(f"  - Source: MedQA Textbooks (English)")
print(f"  - Textbooks: {len(textbooks_df)}")
print(f"  - Total chunks: {len(chunks_df)}")
print(f"  - Embedding model: sentence-transformers/all-MiniLM-L6-v2")
print(f"  - Vector index: FAISS IndexFlatIP (cosine similarity)")
print(f"\nKey Features Implemented:")
print(f"  ✅ Adaptive Query Routing (SIMPLE/MEDIUM/COMPLEX)")
print(f"  ✅ Knowledge Gap Detection (similarity thresholds)")
print(f"  ✅ Safe refusal mechanism (prevents hallucination)")
print(f"\nEvaluation:")
print(f"  - Test queries: {len(eval_df)}")
print(f"  - Gap detection rate: {eval_df['gap_detected'].sum()/len(eval_df)*100:.1f}%")
print(f"  - Avg similarity: {eval_df['top_similarity'].mean():.3f}")


✅ Saved system configuration to 'system_config.json'
✅ Saved KB info to 'kb_info.json'

📄 SUMMARY FOR YOUR REPORT:
Knowledge Base:
  - Source: MedQA Textbooks (English)
  - Textbooks: 10
  - Total chunks: 17457
  - Embedding model: sentence-transformers/all-MiniLM-L6-v2
  - Vector index: FAISS IndexFlatIP (cosine similarity)

Key Features Implemented:
  ✅ Adaptive Query Routing (SIMPLE/MEDIUM/COMPLEX)
  ✅ Knowledge Gap Detection (similarity thresholds)
  ✅ Safe refusal mechanism (prevents hallucination)

Evaluation:
  - Test queries: 10
  - Gap detection rate: 20.0%
  - Avg similarity: 0.547


In [ ]:
# CELL 19: Interactive Demo
def demo_system():
    print("="*80)
    print("🏥 ADAPTIVE RAG MEDICAL Q&A SYSTEM")
    print("="*80)
    print("\nType your medical question (or 'quit' to exit)")
    print("Try queries like:")
    print("  - What is diabetes?")
    print("  - Compare Type 1 and Type 2 diabetes")
    print("  - How do I fix my laptop? (to test gap detection)")
    print()
    
    while True:
        query = input("\n❓ Your question: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Thank you for using the system!")
            break
        
        if not query:
            continue
        
        result = adaptive_rag(query)
        
        print(f"\n🏷️  Complexity: {result['complexity']}")
        print(f"📊 Confidence: {result['top_similarity']:.3f}")
        print(f"🚦 Gap Detected: {result['gap_detected']}")
        print(f"\n💬 Answer:\n{result['answer'][:800]}")
        print("─"*80)

# Uncomment to run interactive demo
demo_system()


🏥 ADAPTIVE RAG MEDICAL Q&A SYSTEM

Type your medical question (or 'quit' to exit)
Try queries like:
  - What is diabetes?
  - Compare Type 1 and Type 2 diabetes
  - How do I fix my laptop? (to test gap detection)




❓ Your question:  does cancer have medicine?



🏷️  Complexity: SIMPLE
📊 Confidence: 0.522
🚦 Gap Detected: False

💬 Answer:
✅ ANSWER (Confidence: 0.52)

Based on retrieved medical textbook evidence:

[1] Source: InternalMed_Harrison.txt (similarity: 0.522)
    with known toxicities. The types of damage from cancer treatment vary. Often, a final common pathway is irreparable damage to DNA. Surgery can create dysfunction, including blind gut loops with absorption problems and loss of function of removed body parts. Radiation may damage end-organ function, for example, loss of potency in prostate cancer pat...

[2] Source: Cell_Biology_Alberts.txt (similarity: 0.522)
    that has a direct carcinogenic action. By causing severe inflammation, chronic infection with parasites and bacteria can also promote the development of some cancers. For example, chronic infection of the stomach with the bacterium Helicobacter py
────────────────────────────────────────────────────────────────────────────────



❓ Your question:  what is vishnu's full name?



🏷️  Complexity: SIMPLE
📊 Confidence: 0.156
🚦 Gap Detected: True

💬 Answer:
❌ KNOWLEDGE GAP DETECTED

Very low similarity (top1=0.156). Query likely outside knowledge base scope.

This question cannot be answered reliably with the current knowledge base. Please consult a qualified medical professional or refer to additional sources.
────────────────────────────────────────────────────────────────────────────────



❓ Your question:  what is the capital of india



🏷️  Complexity: SIMPLE
📊 Confidence: 0.195
🚦 Gap Detected: True

💬 Answer:
❌ KNOWLEDGE GAP DETECTED

Very low similarity (top1=0.195). Query likely outside knowledge base scope.

This question cannot be answered reliably with the current knowledge base. Please consult a qualified medical professional or refer to additional sources.
────────────────────────────────────────────────────────────────────────────────
